# 04 · Baselines
Trivial (majority class) + TF-IDF/LogReg, scored on the golden set against the rule-based classifier.

Production logic: `src/baseline_trivial.py`, `src/baseline_tfidf.py`, `src/eval.py`.

In [1]:
import os, sys
os.environ['PYTHONUTF8'] = '1'
sys.path.insert(0, os.path.abspath('..'))
os.chdir('..')
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, classification_report
from src import pipeline, baseline_trivial, baseline_tfidf
gold = pd.read_csv('data/golden_set.csv')
msgs, y = gold['customer_msg'].tolist(), gold['true_intent'].tolist()
INT = pipeline.INTENTS

In [2]:
preds = {
    'trivial': baseline_trivial.predict(msgs),
    'tfidf_logreg': baseline_tfidf.predict(msgs),
    'rules': [pipeline.classify_intent(m, mode='rules')['intent'] for m in msgs],
}
for name, yp in preds.items():
    acc = accuracy_score(y, yp)
    f1 = f1_score(y, yp, average='macro', labels=INT, zero_division=0)
    print(f'{name:14s} acc={acc:.3f}  macro-F1={f1:.3f}')

trivial        acc=0.275  macro-F1=0.062
tfidf_logreg   acc=0.550  macro-F1=0.533
rules          acc=0.560  macro-F1=0.545


## Per-class report for the rule classifier

In [3]:
print(classification_report(y, preds['rules'], labels=INT, zero_division=0, digits=3))

                   precision    recall  f1-score   support

  billing_payment      0.854     0.636     0.729        55
   account_access      0.800     0.533     0.640        15
       trip_issue      0.321     0.290     0.305        31
  safety_incident      0.800     0.364     0.500        11
   delivery_order      0.579     0.688     0.629        16
service_complaint      0.714     0.323     0.444        31
    general_query      0.422     0.854     0.565        41

         accuracy                          0.560       200
        macro avg      0.641     0.527     0.545       200
     weighted avg      0.632     0.560     0.558       200



## Honesty check
The TF-IDF/LogReg baseline was trained on the rule classifier's **weak labels** over the corpus (no other labels exist besides the test-only golden set). So its near-tie with the rules is *expected* — it distills them. Beating this honestly needs the few-shot LLM path (set a free key, then `python -m src.eval` adds that row). Full write-up in `reports/failure_analysis.md`.

In [4]:
# reproduce the full committed report
from src import eval as ev
ev.main()


Wrote reports\eval_results.md

Summary (macro-F1):
  Trivial (majority class)           0.062
  TF-IDF + LogReg (weak-labeled)     0.533
  Rule-based classifier              0.545
